# Paper Metrics

Recomputes every reported number from the saved per-shot predictions, and emits
LaTeX tables.

### Why this exists

Most of the numbers quoted so far came from printed cell output rather than
saved files. Per-class **precision and recall**, confidence intervals and
significance tests were never written to disk. A paper needs all three, and a
reviewer may ask how a specific figure was obtained.

Everything here derives from three artifacts that *were* saved:

| file | contents |
|---|---|
| `stage4_predictions.csv` | per-stroke prediction, scalar baseline |
| `stage5_predictions.csv` | per-stroke prediction, temporal model |
| `oof_logits.npz` | raw held-out logits + labels, for bootstrap |

Plus the per-fold and detection CSVs for the tolerance and ablation tables.

### What gets produced

1. Main results table, per class, precision / recall / F1 / support, with **bootstrap 95% CIs**
2. Per-fold table with mean ± std
3. **Paired test** of temporal vs scalar across folds
4. Detection tolerance curve
5. Ablation tables
6. LaTeX for all of the above

No GPU. Runs in under a minute.


## 1 · Mount & load

In [1]:
BASE = "/content/drive/MyDrive/tt_coach"

from google.colab import drive
drive.mount('/content/drive')

import json, itertools
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_recall_fscore_support,
)

BASE = Path(BASE); META = BASE/"derived/meta"; M = BASE/"outputs/metrics"
OUT  = BASE/"outputs/paper"; OUT.mkdir(parents=True, exist_ok=True)

CLASSES = ["serve", "attack", "control", "defence"]
C2I = {c: i for i, c in enumerate(CLASSES)}
RNG = np.random.default_rng(42)
N_BOOT = 10_000

s4 = pd.read_csv(M/"stage4_predictions.csv")
s5 = pd.read_csv(M/"stage5_predictions.csv")
print(f"stage 4 (scalar baseline): {len(s4)} strokes")
print(f"stage 5 (temporal model) : {len(s5)} strokes")
print(f"folds: {sorted(s5.fold.unique())}")

oof_path = META/"oof_logits.npz"
HAS_LOGITS = oof_path.exists()
if HAS_LOGITS:
    z = np.load(oof_path)
    print(f"held-out logits: {z['logits'].shape}")

Mounted at /content/drive
stage 4 (scalar baseline): 1432 strokes
stage 5 (temporal model) : 1432 strokes
folds: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
held-out logits: (1432, 4)


## 2 · Bootstrap

Confidence intervals come from resampling the held-out predictions. With 1,432
strokes the interval is meaningful; with 7 folds a normal-approximation CI on
the fold mean would not be.

Resampling is **stratified by fold**, so the CI reflects the same
video-grouped structure as the evaluation. Resampling strokes freely would
treat two strokes from one rally as independent, which they are not.

In [2]:
def macro_f1(y, p):
    return f1_score(y, p, average="macro", zero_division=0)


def bootstrap_ci(y, p, fold, stat=macro_f1, n=N_BOOT, alpha=0.05):
    """Stratified bootstrap CI. Resamples WITHIN each fold."""
    y, p, fold = np.asarray(y), np.asarray(p), np.asarray(fold)
    idx_by_fold = {f: np.where(fold == f)[0] for f in np.unique(fold)}
    vals = np.empty(n)
    for b in range(n):
        pick = np.concatenate([RNG.choice(ix, len(ix), replace=True)
                               for ix in idx_by_fold.values()])
        vals[b] = stat(y[pick], p[pick])
    lo, hi = np.percentile(vals, [100*alpha/2, 100*(1-alpha/2)])
    return float(stat(y, p)), float(lo), float(hi)


def per_class_table(y, p, fold, name=""):
    y, p = np.asarray(y), np.asarray(p)
    pr, rc, f1, sup = precision_recall_fscore_support(
        y, p, labels=range(len(CLASSES)), zero_division=0)
    rows = []
    for i, c in enumerate(CLASSES):
        _, lo, hi = bootstrap_ci(
            y, p, fold, stat=lambda a, b, i=i: f1_score(a == i, b == i,
                                                        zero_division=0))
        rows.append(dict(model=name, cls=c, precision=pr[i], recall=rc[i],
                         f1=f1[i], f1_lo=lo, f1_hi=hi, support=int(sup[i])))
    m, lo, hi = bootstrap_ci(y, p, fold)
    rows.append(dict(model=name, cls="macro", precision=pr.mean(),
                     recall=rc.mean(), f1=m, f1_lo=lo, f1_hi=hi,
                     support=int(sup.sum())))
    return pd.DataFrame(rows)


y4, p4 = s4.shot_class.map(C2I).values, s4.pred.map(C2I).values
y5, p5 = s5.shot_class.map(C2I).values, s5.pred.map(C2I).values

t4 = per_class_table(y4, p4, s4.fold.values, "scalar (LightGBM)")
t5 = per_class_table(y5, p5, s5.fold.values, "temporal (TCN)")
main = pd.concat([t4, t5], ignore_index=True)
main.to_csv(OUT/"table_main.csv", index=False)

print("PER-CLASS, held out, 95% bootstrap CI on F1")
print("=" * 76)
for name, g in main.groupby("model", sort=False):
    print(f"\n{name}")
    for r in g.itertuples():
        ci = f"[{r.f1_lo:.3f}, {r.f1_hi:.3f}]"
        print(f"  {r.cls:<8} P {r.precision:.3f}  R {r.recall:.3f}  "
              f"F1 {r.f1:.3f} {ci:>16}  n={r.support}")

PER-CLASS, held out, 95% bootstrap CI on F1

scalar (LightGBM)
  serve    P 0.920  R 0.870  F1 0.894   [0.866, 0.920]  n=277
  attack   P 0.746  R 0.803  F1 0.773   [0.749, 0.797]  n=650
  control  P 0.740  R 0.724  F1 0.732   [0.689, 0.771]  n=279
  defence  P 0.406  R 0.354  F1 0.378   [0.318, 0.435]  n=226
  macro    P 0.703  R 0.688  F1 0.694   [0.671, 0.717]  n=1432

temporal (TCN)
  serve    P 0.971  R 0.960  F1 0.966   [0.949, 0.980]  n=277
  attack   P 0.843  R 0.766  F1 0.803   [0.778, 0.826]  n=650
  control  P 0.711  R 0.864  F1 0.780   [0.743, 0.815]  n=279
  defence  P 0.504  R 0.509  F1 0.507   [0.450, 0.560]  n=226
  macro    P 0.757  R 0.775  F1 0.764   [0.741, 0.785]  n=1432


## 3 · Per-fold results

The fold spread matters as much as the mean: it shows whether the model
generalises across players or only scores well on average.

In [3]:
def fold_table(df, name):
    y = df.shot_class.map(C2I).values
    p = df.pred.map(C2I).values
    rows = []
    for f in sorted(df.fold.unique()):
        m = (df.fold == f).values
        rows.append(dict(model=name, fold=f, n=int(m.sum()),
                         macro_f1=macro_f1(y[m], p[m]),
                         accuracy=float((y[m] == p[m]).mean()),
                         **{c: f1_score(y[m] == i, p[m] == i, zero_division=0)
                            for i, c in enumerate(CLASSES)}))
    return pd.DataFrame(rows)


f4, f5 = fold_table(s4, "scalar"), fold_table(s5, "temporal")
folds = pd.concat([f4, f5], ignore_index=True)
folds.to_csv(OUT/"table_folds.csv", index=False)

print("PER FOLD")
print("=" * 76)
for name, g in folds.groupby("model", sort=False):
    print(f"\n{name}")
    print(g[["fold","n","macro_f1","accuracy"]+CLASSES].to_string(
        index=False, float_format=lambda x: f"{x:.3f}"))
    w = g.n/g.n.sum()
    print(f"  weighted macro-F1 {(g.macro_f1*w).sum():.3f}   "
          f"unweighted {g.macro_f1.mean():.3f} +/- {g.macro_f1.std():.3f}")

PER FOLD

scalar
fold   n  macro_f1  accuracy  serve  attack  control  defence
   A 160     0.636     0.669  0.844   0.779    0.649    0.273
   B 385     0.782     0.831  0.949   0.852    0.816    0.512
   C 152     0.695     0.730  0.820   0.818    0.653    0.491
   D 170     0.497     0.529  0.766   0.618    0.493    0.109
   E 248     0.609     0.685  0.887   0.644    0.793    0.113
   F 155     0.799     0.800  0.951   0.818    0.875    0.552
   G 162     0.729     0.759  0.906   0.805    0.667    0.538
  weighted macro-F1 0.689   unweighted 0.678 +/- 0.106

temporal
fold   n  macro_f1  accuracy  serve  attack  control  defence
   A 160     0.692     0.675  0.980   0.722    0.815    0.250
   B 385     0.836     0.868  0.983   0.875    0.902    0.583
   C 152     0.786     0.789  0.984   0.845    0.708    0.608
   D 170     0.748     0.735  0.929   0.800    0.702    0.560
   E 248     0.636     0.730  0.939   0.644    0.778    0.182
   F 155     0.842     0.845  0.952   0.859    0.8

## 4 · Is the temporal model actually better?

Two tests, because neither alone is convincing.

**Paired across folds.** The natural test, but with n=7 the smallest p a
one-sided Wilcoxon signed-rank can return is $2^{-7} = 0.0078$, even if the
temporal model wins every fold. Low power, and worth saying so rather than
reporting a p-value as though it settles the question.

**Bootstrap on the difference.** Resamples the paired predictions directly,
which uses all 1,432 strokes instead of 7 summary numbers.

In [4]:
a = f4.set_index("fold").macro_f1
b = f5.set_index("fold").macro_f1
common = sorted(set(a.index) & set(b.index))
a, b = a.loc[common].values, b.loc[common].values
d = b - a

print("PAIRED ACROSS FOLDS")
print("=" * 70)
for f, x, y_, dd in zip(common, a, b, d):
    print(f"  {f}   scalar {x:.3f}   temporal {y_:.3f}   {dd:+.3f}")
print(f"\n  mean difference {d.mean():+.4f} +/- {d.std(ddof=1):.4f}")
print(f"  temporal better in {(d>0).sum()}/{len(d)} folds")

w = stats.wilcoxon(b, a, alternative="greater")
t = stats.ttest_rel(b, a, alternative="greater")
print(f"\n  Wilcoxon signed-rank  W={w.statistic:.1f}  p={w.pvalue:.4f}")
print(f"  paired t-test         t={t.statistic:.3f}  p={t.pvalue:.4f}")
print(f"""
  With n={len(d)} folds the smallest p Wilcoxon can return is
  {2**-len(d):.4f}. Report these alongside the bootstrap below rather than
  on their own.""")

# --- paired bootstrap over strokes -------------------------------------------
key = "stroke_id" if "stroke_id" in s4.columns and "stroke_id" in s5.columns else None
if key:
    j = s4[[key,"fold","shot_class","pred"]].merge(
        s5[[key,"pred"]], on=key, suffixes=("_4","_5"))
    yy = j.shot_class.map(C2I).values
    pp4 = j.pred_4.map(C2I).values
    pp5 = j.pred_5.map(C2I).values
    ff  = j.fold.values
    print(f"\nPAIRED BOOTSTRAP over {len(j)} strokes present in both")
    print("=" * 70)
    idx_by_fold = {f: np.where(ff == f)[0] for f in np.unique(ff)}
    diffs = np.empty(N_BOOT)
    for i in range(N_BOOT):
        pick = np.concatenate([RNG.choice(ix, len(ix), replace=True)
                               for ix in idx_by_fold.values()])
        diffs[i] = macro_f1(yy[pick], pp5[pick]) - macro_f1(yy[pick], pp4[pick])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    obs = macro_f1(yy, pp5) - macro_f1(yy, pp4)
    print(f"  observed difference {obs:+.4f}")
    print(f"  95% CI [{lo:+.4f}, {hi:+.4f}]")
    print(f"  P(temporal > scalar) = {(diffs > 0).mean():.4f}")
    print(f"  -> {'CI excludes zero' if lo > 0 else 'CI includes zero'}")
else:
    print("\n  stroke_id missing from one file — paired bootstrap skipped")

PAIRED ACROSS FOLDS
  A   scalar 0.636   temporal 0.692   +0.055
  B   scalar 0.782   temporal 0.836   +0.053
  C   scalar 0.695   temporal 0.786   +0.091
  D   scalar 0.497   temporal 0.748   +0.251
  E   scalar 0.609   temporal 0.636   +0.026
  F   scalar 0.799   temporal 0.842   +0.043
  G   scalar 0.729   temporal 0.712   -0.017

  mean difference +0.0718 +/- 0.0855
  temporal better in 6/7 folds

  Wilcoxon signed-rank  W=27.0  p=0.0156
  paired t-test         t=2.221  p=0.0341

  With n=7 folds the smallest p Wilcoxon can return is
  0.0078. Report these alongside the bootstrap below rather than
  on their own.

PAIRED BOOTSTRAP over 1432 strokes present in both
  observed difference +0.0692
  95% CI [+0.0436, +0.0944]
  P(temporal > scalar) = 1.0000
  -> CI excludes zero


## 5 · Confusion matrices

In [5]:
for name, y, p in (("scalar", y4, p4), ("temporal", y5, p5)):
    cm = confusion_matrix(y, p, labels=range(len(CLASSES)))
    df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
    df["recall"] = (np.diag(cm)/cm.sum(1)).round(3)
    print(f"\n{name.upper()}")
    print(df.to_string())
    df.to_csv(OUT/f"confusion_{name}.csv")

# the pair that resists every intervention
cm = confusion_matrix(y5, p5, labels=range(4))
ad = cm[1,3] + cm[3,1]
print(f"\nattack <-> defence confusion: {ad} of "
      f"{cm[1].sum()+cm[3].sum()} ({100*ad/(cm[1].sum()+cm[3].sum()):.1f}%)")


SCALAR
         serve  attack  control  defence  recall
serve      241      24        1       11   0.870
attack      11     522       43       74   0.803
control      2      43      202       32   0.724
defence      8     111       27       80   0.354

TEMPORAL
         serve  attack  control  defence  recall
serve      266       2        5        4   0.960
attack       4     498       63       85   0.766
control      3      11      241       24   0.864
defence      1      80       30      115   0.509

attack <-> defence confusion: 165 of 876 (18.8%)


## 6 · Detection and tolerance

The tolerance sweep is the evidence for respecifying the target. It shows
classifier accuracy is nearly flat across a factor-of-two change in detection
precision, which is why ±8 frames rather than ±5 is the operationally correct
bar.

In [6]:
for f, label in (("stage6_tolerance.csv", "detection vs tolerance"),
                 ("stage7_tolerance_sweep.csv", "end-to-end vs tolerance"),
                 ("stage6_per_video.csv", "detection per video"),
                 ("stage5_aux_sweep.csv", "auxiliary weight sweep"),
                 ("stage4_taxonomy_comparison.csv", "taxonomy variants")):
    p = M/f
    if not p.exists():
        print(f"  missing: {f}"); continue
    d = pd.read_csv(p)
    print(f"\n{label.upper()}  ({f})")
    print(d.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    d.to_csv(OUT/f, index=False)


DETECTION VS TOLERANCE  (stage6_tolerance.csv)
 tol_frames  tol_ms  precision  recall    f1
          1       8      0.392   0.352 0.370
          2      17      0.572   0.514 0.542
          3      25      0.685   0.615 0.648
          5      42      0.820   0.736 0.776
          8      67      0.886   0.796 0.839
         12     100      0.904   0.812 0.855
         20     167      0.916   0.822 0.867

END-TO-END VS TOLERANCE  (stage7_tolerance_sweep.csv)
 tol  ms  det_f1  cls_acc  cls_macro_f1  e2e_f1    n
   1   8   0.381    0.510         0.470   0.194  529
   2  17   0.563    0.501         0.470   0.282  782
   3  25   0.677    0.496         0.471   0.336  941
   5  42   0.797    0.497         0.483   0.396 1107
   8  67   0.853    0.491         0.481   0.419 1185
  12 100   0.868    0.492         0.483   0.427 1205
  20 167   0.879    0.493         0.485   0.433 1221

DETECTION PER VIDEO  (stage6_per_video.csv)
video_id fold   n  tp  fp  fn  precision  recall    f1  side_acc
  g

## 7 · LaTeX tables

Written to `outputs/paper/`. `booktabs` formatting, ready to include.

In [7]:
def tex_main(df, path, caption, label):
    lines = [r"\begin{table}[t]", r"\centering",
             r"\caption{"+caption+"}", r"\label{tab:"+label+"}",
             r"\begin{tabular}{llrrrr}", r"\toprule",
             r"Model & Class & P & R & F1 (95\% CI) & $n$ \\", r"\midrule"]
    for name, g in df.groupby("model", sort=False):
        for i, r in enumerate(g.itertuples()):
            m = name if i == 0 else ""
            cls = r"\textbf{macro}" if r.cls == "macro" else r.cls
            if r.cls == "macro":
                lines.append(r"\cmidrule(lr){2-6}")
            lines.append(
                f"{m} & {cls} & {r.precision:.3f} & {r.recall:.3f} & "
                f"{r.f1:.3f} [{r.f1_lo:.3f}, {r.f1_hi:.3f}] & {r.support} \\\\")
        lines.append(r"\midrule")
    lines[-1] = r"\bottomrule"
    lines += [r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(lines))
    return "\n".join(lines)


def tex_folds(df, path, caption, label):
    lines = [r"\begin{table}[t]", r"\centering",
             r"\caption{"+caption+"}", r"\label{tab:"+label+"}",
             r"\begin{tabular}{lrrr}", r"\toprule",
             r"Fold & $n$ & Scalar & Temporal \\", r"\midrule"]
    a = df[df.model=="scalar"].set_index("fold")
    b = df[df.model=="temporal"].set_index("fold")
    for f in sorted(set(a.index) & set(b.index)):
        lines.append(f"{f} & {int(b.loc[f,'n'])} & "
                     f"{a.loc[f,'macro_f1']:.3f} & {b.loc[f,'macro_f1']:.3f} \\\\")
    lines.append(r"\midrule")
    lines.append(f"Mean & & {a.macro_f1.mean():.3f} $\\pm$ {a.macro_f1.std():.3f} & "
                 f"{b.macro_f1.mean():.3f} $\\pm$ {b.macro_f1.std():.3f} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(lines))
    return "\n".join(lines)


t1 = tex_main(main, OUT/"table_main.tex",
    "Stroke classification, 7-fold cross-validation grouped by video. "
    "Confidence intervals from a stratified bootstrap ($10{,}000$ resamples).",
    "main")
t2 = tex_folds(folds, OUT/"table_folds.tex",
    "Per-fold macro-F1. Each fold holds out one or more complete videos.",
    "folds")
print(t1); print(); print(t2)
print(f"\n-> {OUT}")

\begin{table}[t]
\centering
\caption{Stroke classification, 7-fold cross-validation grouped by video. Confidence intervals from a stratified bootstrap ($10{,}000$ resamples).}
\label{tab:main}
\begin{tabular}{llrrrr}
\toprule
Model & Class & P & R & F1 (95\% CI) & $n$ \\
\midrule
scalar (LightGBM) & serve & 0.920 & 0.870 & 0.894 [0.866, 0.920] & 277 \\
 & attack & 0.746 & 0.803 & 0.773 [0.749, 0.797] & 650 \\
 & control & 0.740 & 0.724 & 0.732 [0.689, 0.771] & 279 \\
 & defence & 0.406 & 0.354 & 0.378 [0.318, 0.435] & 226 \\
\cmidrule(lr){2-6}
 & \textbf{macro} & 0.703 & 0.688 & 0.694 [0.671, 0.717] & 1432 \\
\midrule
temporal (TCN) & serve & 0.971 & 0.960 & 0.966 [0.949, 0.980] & 277 \\
 & attack & 0.843 & 0.766 & 0.803 [0.778, 0.826] & 650 \\
 & control & 0.711 & 0.864 & 0.780 [0.743, 0.815] & 279 \\
 & defence & 0.504 & 0.509 & 0.507 [0.450, 0.560] & 226 \\
\cmidrule(lr){2-6}
 & \textbf{macro} & 0.757 & 0.775 & 0.764 [0.741, 0.785] & 1432 \\
\bottomrule
\end{tabular}
\end{table}

\b

## 8 · Summary for the abstract

In [8]:
m5 = main[(main.model.str.startswith("temporal")) & (main.cls=="macro")].iloc[0]
m4 = main[(main.model.str.startswith("scalar")) & (main.cls=="macro")].iloc[0]
print("=" * 70)
print(f"  scalar baseline    macro-F1 {m4.f1:.3f} [{m4.f1_lo:.3f}, {m4.f1_hi:.3f}]")
print(f"  temporal model     macro-F1 {m5.f1:.3f} [{m5.f1_lo:.3f}, {m5.f1_hi:.3f}]")
print(f"  improvement        {m5.f1-m4.f1:+.3f}")
print()
for c in CLASSES:
    r = main[(main.model.str.startswith("temporal")) & (main.cls==c)].iloc[0]
    tag = "coachable" if r.precision >= 0.85 else "SUPPRESSED"
    print(f"  {c:<8} P {r.precision:.3f}  R {r.recall:.3f}  F1 {r.f1:.3f}   {tag}")
print("=" * 70)
print("""
  Reminder for the write-up: these are held-out cross-validation numbers.
  The shipped models were trained on all videos, so nothing computed on
  those same videos is reportable. Quote these.""")

  scalar baseline    macro-F1 0.694 [0.671, 0.717]
  temporal model     macro-F1 0.764 [0.741, 0.785]
  improvement        +0.069

  serve    P 0.971  R 0.960  F1 0.966   coachable
  attack   P 0.843  R 0.766  F1 0.803   SUPPRESSED
  control  P 0.711  R 0.864  F1 0.780   SUPPRESSED
  defence  P 0.504  R 0.509  F1 0.507   SUPPRESSED

  Reminder for the write-up: these are held-out cross-validation numbers.
  The shipped models were trained on all videos, so nothing computed on
  those same videos is reportable. Quote these.


---
## Outputs

| file | |
|---|---|
| `table_main.csv` / `.tex` | per-class P/R/F1 with bootstrap CIs |
| `table_folds.csv` / `.tex` | per-fold macro-F1 |
| `confusion_*.csv` | confusion matrices |
| copies of the sweep CSVs | tolerance, ablation, taxonomy |

**On the significance tests:** with 7 folds, a one-sided Wilcoxon signed-rank
cannot return p below 0.0078 however large the effect. Report it alongside the paired
bootstrap over strokes, which uses all 1,432 predictions rather than 7 summary
numbers, and say plainly that fold-level power is limited.
